# Issue #253 — M×3 physical schema prototype

This notebook builds and profiles the smallest end-to-end external package. The model key is `issue_code × match_idx × outcome_code`; historical proxy and strict-forward observations remain separate namespaces, and settlement labels are physically separate.

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd

repo = Path.cwd()
sys.path.insert(0, str(repo / 'src'))
from physical_schema import build_representative_package, load_feature_cells, resolve_data_root, verify_package

data_root = resolve_data_root(os.environ.get('BEIJING_LOT_DATA_ROOT'))
data_root

PosixPath('/home/drwho1985/workspace/soccer/beijing-lot/data')

In [2]:
package = build_representative_package(data_root)
verification = verify_package(package)
pd.Series(verification, name='value').to_frame()

,value
status,PASS
issue_slots,8
issues,3
observations,16
feature_cells,60
available_feature_cells,58
missing_feature_cells,2
settlement_labels,4
namespace_feature_cells,"{'historical_proxy': 36, 'strict_forward': 24}"
feature_name_cells,"{'bj_sp': 12, 'eu_decimal_odds': 48}"


In [3]:
features = load_feature_cells(package)
profile = (features.groupby(['namespace', 'snapshot_horizon', 'feature_name'], dropna=False)
           .agg(cells=('value', 'size'), available=('availability_mask', 'sum'), slots=('issue_slot_id', 'nunique'))
           .reset_index())
assert not {'result_code', 'winning_sp', 'settled_at_utc'}.intersection(features.columns)
profile

,namespace,snapshot_horizon,feature_name,cells,available,slots
0,historical_proxy,10h,eu_decimal_odds,12,12,4
1,historical_proxy,1h,eu_decimal_odds,12,12,4
2,historical_proxy,initial,eu_decimal_odds,12,12,4
3,strict_forward,poll,bj_sp,12,10,4
4,strict_forward,poll,eu_decimal_odds,12,12,4


## Interpretation

This is an interface proof, not a performance result or a full-history rebuild. Missing SP cells remain null with explicit masks. The current strict-forward sample is bounded by poll-start/current-schedule semantics and live URL-expect identity remains noncanonical.